# 03 Merge and Clean — Graphic Novels

Goal:
Clean and filter the graphic novel dataset.

Input:
`../Data/Clean/graphic_novels_with_descriptions.csv`

Output:
`../Data/Clean/merged_graphic_novels.csv`

In [2]:
import pandas as pd
import re
from pathlib import Path

In [3]:
input_path = Path("../Data/Clean/graphic_novels_with_descriptions.csv")

df = pd.read_csv(input_path)

df.shape

(5609, 11)

In [4]:
df.head()

,search_term,ol_key,title,author,first_publish_year,publisher,language,subject,isbn,cover_i,description
0,graphic novel,/works/OL24244816W,Allergic,"Megan Wagner Lloyd, Michelle Mee Nutter",2021.0,NaN,eng,NaN,NaN,10724638.0,"At home, Maggie is the odd one out. Her parent..."
1,graphic novel,/works/OL24793569W,The Alchemist Graphic Novel,Paulo Coelho,2010.0,NaN,eng,NaN,NaN,11556106.0,NaN
2,graphic novel,/works/OL15902631W,The Kite Runner--the graphic novel,Khaled Hosseini,2011.0,NaN,"eng, ara",NaN,NaN,8063795.0,"""The spellbinding story of the unlikely and in..."
3,graphic novel,/works/OL16793507W,Drama,Raina Telgemeier,2012.0,NaN,"spa, fre, eng",NaN,NaN,12360868.0,Callie loves theater. And while she would tota...
4,graphic novel,/works/OL28062430W,Moon Rising,Tui T. Sutherland,2022.0,NaN,eng,NaN,NaN,15099568.0,NaN


In [5]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

df.columns

Index(['search_term', 'ol_key', 'title', 'author', 'first_publish_year',
       'publisher', 'language', 'subject', 'isbn', 'cover_i', 'description'],
      dtype='object')

In [6]:
# Keep only relevant columns for analysis
columns_to_keep = [
    "title",
    "author",
    "first_publish_year",
    "publisher",
    "language",
    "subject",
    "isbn",
    "cover_i",
    "description",
    "search_term",
    "ol_key"
]

df = df[[col for col in columns_to_keep if col in df.columns]]

df.head()

,title,author,first_publish_year,publisher,language,subject,isbn,cover_i,description,search_term,ol_key
0,Allergic,"Megan Wagner Lloyd, Michelle Mee Nutter",2021.0,NaN,eng,NaN,NaN,10724638.0,"At home, Maggie is the odd one out. Her parent...",graphic novel,/works/OL24244816W
1,The Alchemist Graphic Novel,Paulo Coelho,2010.0,NaN,eng,NaN,NaN,11556106.0,NaN,graphic novel,/works/OL24793569W
2,The Kite Runner--the graphic novel,Khaled Hosseini,2011.0,NaN,"eng, ara",NaN,NaN,8063795.0,"""The spellbinding story of the unlikely and in...",graphic novel,/works/OL15902631W
3,Drama,Raina Telgemeier,2012.0,NaN,"spa, fre, eng",NaN,NaN,12360868.0,Callie loves theater. And while she would tota...,graphic novel,/works/OL16793507W
4,Moon Rising,Tui T. Sutherland,2022.0,NaN,eng,NaN,NaN,15099568.0,NaN,graphic novel,/works/OL28062430W


In [7]:
df = df.dropna(subset=["title", "author", "description"])

df.shape

(2195, 11)

In [8]:
df = df.drop_duplicates(subset=["title", "author"])

df.shape

(2195, 11)

In [9]:
# Create a combined text column for filtering
df["filter_text"] = (
    df["title"].fillna("") + " " +
    df["author"].fillna("") + " " +
    df["subject"].fillna("") + " " +
    df["description"].fillna("")
).str.lower()

In [10]:
# Define keywords to identify graphic novels
include_terms = [
    "graphic novel",
    "graphic novels",
    "sequential art",
    "graphic memoir",
    "illustrated",
    "cartoon",
    "visual storytelling"
]

In [ ]:
# Filter the DataFrame to include only rows that contain any of the include terms

exclude_terms = [
    # educational / instructional
    "how to",
    "guide",
    "manual",
    "workbook",
    "textbook",
    "lesson",
    "curriculum",
    "course",
    "introduction to",
    "study of",

    # making comics
    "write your own",
    "how to draw",
    "drawing",
    "learn to draw",
    "making comics",
    "create comics",
    "comic creation",

    # theory / analysis
    "history of comics",
    "theory",
    "analysis",
    "criticism",
    "review",
    "essays on",
    "understanding comics",
    "inventing comics",

    # professions
    "animation",
    "working in animation",

    # general noise
    "coloring book",
    "activity book",
    "children's",
    "children",
    "teacher",
    "students"
]

In [12]:
include_pattern = "|".join(include_terms)
exclude_pattern = "|".join(exclude_terms)

df = df[df["filter_text"].str.contains(include_pattern, na=False)]
df = df[~df["filter_text"].str.contains(exclude_pattern, na=False)]

df.shape

(386, 12)

In [19]:
narrative_keywords = [
    "story",
    "tale",
    "novel",
    "memoir",
    "journey",
    "life",
    "family",
    "war",
    "history",
    "love",
    "childhood",
    "coming of age"
]

df = df[
    df["clean_description"].apply(
        lambda x: any(k in x for k in narrative_keywords)
    )
]

df.shape

(351, 16)

In [20]:
df[["title", "author"]].sample(20)

,title,author
967,Foundations of Chinese civilization,"Liu, Jing (Author of graphic novels)"
2221,William Shakespeare's The winter's tale,Vincent Goodwin
458,Graphic Novels As Philosophy,Jeff McLaughlin
1113,The Best We Could Do,Thi Bui
2010,Gothicka,Victoria Nelson
4585,Bordados / Embroideries,"Marjane Satrapi, Dan Franklin, Carlos Mayor Or..."
1916,Wuvable Oaf,Ed Luce
2110,The strange case of Dr. Jekyll and Mr. Hyde,"C. E. L. Welsh, Lalit Sharma"
2669,Poe,Gareth Hinds
4583,Henry Ford and the Model T,Michael O'Hearn


In [21]:
df[df["author"].str.contains("satrapi", case=False, na=False)]

,title,author,first_publish_year,publisher,language,subject,isbn,cover_i,description,search_term,ol_key,filter_text,clean_description,clean_subject,combined_text,cover_url
4585,Bordados / Embroideries,"Marjane Satrapi, Dan Franklin, Carlos Mayor Or...",2004.0,NaN,"pol, cat, eng, spa, fre",NaN,NaN,1992636.0,"A collection of stories and anecdotes, told in...",graphic biography,/works/OL5735173W,"bordados / embroideries marjane satrapi, dan f...",a collection of stories and anecdotes told in ...,,"Bordados / Embroideries Marjane Satrapi, Dan F...",https://covers.openlibrary.org/b/id/1992636-M.jpg


In [22]:
df[df["author"].str.contains("eisner", case=False, na=False)]

,title,author,first_publish_year,publisher,language,subject,isbn,cover_i,description,search_term,ol_key,filter_text,clean_description,clean_subject,combined_text,cover_url
4763,"Life, in Pictures",Will Eisner,2007.0,NaN,eng,NaN,NaN,1194796.0,A collection of three graphic novels and two s...,graphic biography,/works/OL1871512W,"life, in pictures will eisner a collection of...",a collection of three graphic novels and two s...,,"Life, in Pictures Will Eisner a collection of...",https://covers.openlibrary.org/b/id/1194796-M.jpg


In [23]:
df[df["author"].str.contains("spiegelman", case=False, na=False)]

,title,author,first_publish_year,publisher,language,subject,isbn,cover_i,description,search_term,ol_key,filter_text,clean_description,clean_subject,combined_text,cover_url
220,Maus I,Art Spiegelman,1986.0,NaN,"fre, pol, ita, cat, eng",NaN,NaN,10210168.0,A story of a Jewish survivor of Hitler's Europ...,graphic novel,/works/OL2056818W,maus i art spiegelman a story of a jewish sur...,a story of a jewish survivor of hitler s europ...,,Maus I Art Spiegelman a story of a jewish sur...,https://covers.openlibrary.org/b/id/10210168-M...
5420,"An Anthology of Graphic Fiction, Cartoons, and...","Ivan Brunetti, Robert Crumb, Kim Deitch, Art S...",2006.0,NaN,eng,NaN,NaN,162741.0,"""Comic artist Ivan Brunetti, the creator of Sc...",graphic history,/works/OL8860942W,"an anthology of graphic fiction, cartoons, and...",comic artist ivan brunetti the creator of schi...,,"An Anthology of Graphic Fiction, Cartoons, and...",https://covers.openlibrary.org/b/id/162741-M.jpg


In [24]:
# Drop the filter_text column as it's no longer needed
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [25]:
# Create cleaned text columns for description and subject
df["clean_description"] = df["description"].apply(clean_text)
df["clean_subject"] = df["subject"].apply(clean_text)

In [26]:
# Create a combined text column for filtering
df["combined_text"] = (
    df["title"].fillna("") + " " +
    df["author"].fillna("") + " " +
    df["clean_subject"].fillna("") + " " +
    df["clean_description"].fillna("")
)

In [27]:
df["cover_url"] = df["cover_i"].apply(
    lambda x: f"https://covers.openlibrary.org/b/id/{int(x)}-M.jpg" if pd.notna(x) else None
)

In [28]:
df[["title", "author", "first_publish_year", "description"]].head()

,title,author,first_publish_year,description
2,The Kite Runner--the graphic novel,Khaled Hosseini,2011.0,"""The spellbinding story of the unlikely and in..."
11,The dragonet prophecy [graphic novel],Barry Deutsch,2018.0,Determined to end a generations-long war among...
14,The hidden kingdom [graphic novel],Barry Deutsch,2000.0,"Write your legend, draw your destiny, and take..."
25,Artemis Fowl. The Graphic Novel,"Eoin Colfer, Andrew Donkin, Michael Moreci, St...",2007.0,"In 2001, audiences first met and fell in love ..."
35,Monster High,Heather Nuhfer,2014.0,Romance is in the air in this brand-new Monste...


In [29]:
# Save the cleaned DataFrame to a new CSV file
output_path = Path("../Data/Clean/merged_graphic_novels.csv")

df.to_csv(output_path, index=False)

print(f"Saved clean data to: {output_path}")
print(f"Final rows: {len(df)}")

Saved clean data to: ../Data/Clean/merged_graphic_novels.csv
Final rows: 351
